In [14]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv('GroqAPIKey')

### **STEP-1** INSTALL THE PACKAGES - We already have the 'Groq' package insatlled.

### **STEP-2** IMPORT THE PACKAGES

In [15]:
from groq import Groq
from openai import OpenAI
import json

In [16]:
#client = Groq(api_key=api_key)
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

In [17]:
message = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"How much is 345 divided by 5"
    }
]


response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=message
)

print(response)
print(f"Answer is::: {response.choices[0].message.content}")

ChatCompletion(id='chatcmpl-ee576cfe-bc25-48e0-839e-dad7a6cae4ba', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='345 divided by 5 equals **69**.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User asks simple math: 345/5 = 69. So answer 69.'))], created=1783403537, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_0b0e15f695', usage=CompletionUsage(completion_tokens=38, prompt_tokens=98, total_tokens=136, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=19, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.319369282, prompt_time=0.004412097, completion_time=0.078584331, total_time=0.082996428), usage_breakdown=None, x_groq={'id': 'req_01kwxj4rg5f8m8nq912sm1sctw', 'seed': 1637293622})
Answer is::: 345 divided by 5 equals *

In [18]:
# This will fail, because, we have given a system promt to agent saying to use tools when needed, but actually, we haven't provided any tools. So, for my quest, it actually need a tool (whether api). Since, we haven't provided tools, it will not be able to connect. So it returned with msg: "Tool choice is none, but model called a tool".
#Here, a point to be noted is for messageUser, it didn't fail, because here we have removed the system prompt. We just used the LLM from user perspective, and didn't give any persona.
messageSystem = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"What is the temperature in Paris no? "
    }
]

messageUser = [
    
    {
        "role":"user",
        "content":"What is the temperature in Paris no? "
    }
]


responseUser = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageUser
)

print(f"responseUser:: {responseUser}")
print(f"Answer is::: {responseUser.choices[0].message.content}")

responseSystem = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageSystem
)


print(f"responseSystem:: {responseSystem}")
print(f"Answer is::: {responseSystem.choices[0].message.content}")

responseUser:: ChatCompletion(id='chatcmpl-86dfc769-16bc-4fb8-ba28-74314d5ac7fd', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m not able to retrieve real‑time weather data, so I can’t give you the current temperature in Paris right now. For the most up‑to‑date forecast, you can check a weather website or app such as\u202fWeather.com, AccuWeather, or simply search “Paris weather” in your preferred search engine. If you meant something else by “no,” just let me know!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User asks: "What is the temperature in Paris no?" Probably they want current temperature in Paris. As an AI without internet access, we can\'t provide real-time data. We should respond that we cannot fetch real-time data, suggest checking a weather service. Also maybe ask for clarification about "no"? Might be typo. We should be concise.'))], created=178340

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "web.run", "arguments": {\n  "id": "search1",\n  "action": "search",\n  "parameters": {\n    "query": "current temperature Paris"\n  }\n}}'}}

### Creating Functions
***Syntax*** : def function_name(parameter: Type) -> ReturnType:

In [19]:
def calculator(expression:str) -> str:
    #eval will evaluate the expression.
    result = eval(expression)
    return str(result)


res = calculator("45*67+23")
print(f"45*67+23 :: {res}")
print(f"type of res: {type(res)}")
    

45*67+23 :: 3038
type of res: <class 'str'>


In [20]:
def mockWhetherData(city:str) -> str:
    whetherData ={
        "chennai":"39c, drizzling",
        "mumbai":"34c, Polluted and dirty",
        "delhi":"29c, Hevealy polluted, bad AQI",
        "kerala":"22c, Plesent, calm and fresh whether"
    }

    if city in whetherData:
        return whetherData.get(city)
    
city='delhi'
positiveTemp = mockWhetherData(city)
print(f"Whether in {city}: {positiveTemp}")

city='bangalore'
negativeTemp = mockWhetherData(city)
print(f"Whether in {city}: {negativeTemp}")


    

Whether in delhi: 29c, Hevealy polluted, bad AQI
Whether in bangalore: None


### The above are the tools that we were talking about. These are the hands that we give to our agents to perform tasks.

### HOW TIO FEED THESE TOOLS TO OUR AGENTS?
We need to create a json explaining our function as a tool to the agent.

In [21]:
tools=[
    {
        "type":"function",
        "function":{
            "name":"calculator", #func name
            "description":"A function to evaluate a mathematical expression. use this for math computions.", #"Returns recipe of a dish."
            "parameters":{
                "type":"object",
                "properties":{
                    "expression":{
                    "type":"string",
                    "description":"The expression to evaluate. Eg: '(45+78)*167'"
                    }
                },
                "required":["expression"]               
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"mockWhetherData", #func name
            "description":"A function to which returns the whether details of the city. Use this for whether related data.",
            "parameters":{
                "type":"object",
                "properties":{
                    "city":{
                    "type":"string",
                    "description":"This is the cuty for which whether details has to be returned. Eg: 'Chennai'"
                    }
                },
                "required":["city"]               
            }
        }
    }
]

In [22]:
print(f"Tools defined for our agent: {tools}")

Tools defined for our agent: [{'type': 'function', 'function': {'name': 'calculator', 'description': 'A function to evaluate a mathematical expression. use this for math computions.', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string', 'description': "The expression to evaluate. Eg: '(45+78)*167'"}}, 'required': ['expression']}}}, {'type': 'function', 'function': {'name': 'mockWhetherData', 'description': 'A function to which returns the whether details of the city. Use this for whether related data.', 'parameters': {'type': 'object', 'properties': {'city': {'type': 'string', 'description': "This is the cuty for which whether details has to be returned. Eg: 'Chennai'"}}, 'required': ['city']}}}]


### Calling LLM with tools defined.

In [23]:
messageExp = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"How much is 345 divided by 5"
    }
]

messageWhether = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"What is the temperature in Paris no? "
    }
]

respwithTools = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageExp,
    tools=tools
)

#If you see below, our finish_reason=""tool_calls, meaning we have a tool that does our work. So, the agent didn't give any result.
print(f"respwithTools choice:: {respwithTools.choices[0]}")
print(f"respwithTools choice:: {respwithTools.choices[0].message.content}")

''' Here, even b4, w/o using tools, we received ans. But with tools, it is acting differently. Why???
Modern reasoning models are trained to prefer tools when an appropriate tool exists.

Think of the model's decision process as:
User asks question -> Do I have a suitable tool? -> YES -> Use the tool -> NO -> Answer directly.

Our tolls matching with the user query heavily depends on the Function name, Description (most important), Parameter names, Parameter descriptions, The user's query, The conversation history. We need to give proper description, otherwise, the model will not pick it up. '''

respwithTools choice:: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_0c9bafa2-f7a2-43bc-a117-75d81376f906', function=Function(arguments='{"expression":"345/5"}', name='calculator'), type='function')], reasoning="User asks a simple calculation: 345 / 5 = 69. Let's compute using tool."))
respwithTools choice:: None


" Here, even b4, w/o using tools, we received ans. But with tools, it is acting differently. Why???\nModern reasoning models are trained to prefer tools when an appropriate tool exists.\n\nThink of the model's decision process as:\nUser asks question -> Do I have a suitable tool? -> YES -> Use the tool -> NO -> Answer directly.\n\nOur tolls matching with the user query heavily depends on the Function name, Description (most important), Parameter names, Parameter descriptions, The user's query, The conversation history. We need to give proper description, otherwise, the model will not pick it up. "

In [24]:
print(f"Finish Reason: {respwithTools.choices[0].finish_reason}")
print(f"Tool calls: {respwithTools.choices[0].message.tool_calls}")

print("-------LLM DECISION---------")
toolCalls=respwithTools.choices[0].message.tool_calls

for i in range(0,len(toolCalls)):
    tool = toolCalls[i]
    print(f"Tool name: {tool.function.name}")
    print(f"Arguments: {tool.function.arguments}")
    print(f"tool id: {tool.id}")


Finish Reason: tool_calls
Tool calls: [ChatCompletionMessageFunctionToolCall(id='fc_0c9bafa2-f7a2-43bc-a117-75d81376f906', function=Function(arguments='{"expression":"345/5"}', name='calculator'), type='function')]
-------LLM DECISION---------
Tool name: calculator
Arguments: {"expression":"345/5"}
tool id: fc_0c9bafa2-f7a2-43bc-a117-75d81376f906


In [25]:
availableTools=   {
        "calculator":calculator,
        "mockWhetherData":mockWhetherData
    }

arguments = json.loads(tool.function.arguments)
print(arguments)

{'expression': '345/5'}


In [26]:
result = availableTools[tool.function.name](**arguments)
print(result)

69.0


### Feeding the tool response to LLM.

In [28]:
messageExp.append(respwithTools.choices[0].message)
messageExp.append({
    "role":"tool",
    "tool_call_id":tool.id,
    "content":result
})

finalrespTools = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageExp,
    tools=tools
)

#print(f"finalrespTools:: {finalrespTools}")
print(f"finalrespTools:: {finalrespTools.choices[0].message.content}")

finalrespTools:: 345 divided by 5 equals **69**.
